# Stellar-Qwen3-8B-ARM — RM-Bench Evaluation

- **Model:** [`Stellar-lab/Stellar-Qwen3-8B-ARM`](https://huggingface.co/Stellar-lab/Stellar-Qwen3-8B-ARM)
- **Dataset:** [THU-KEG/RM-Bench](https://huggingface.co/datasets/THU-KEG/RM-Bench) (1,327 prompts × 6 responses / 3×3 style matrix)
- **Reported Result:** **74.88%** Overall Average (`82.69%` Easy | `76.44%` Normal | `65.51%` Hard)
- **Hardware:** Requires a CUDA GPU (`>=2` GPUs run FP16 via `device_map="auto"`; `1` GPU runs 4-bit NF4 quantization)

In [1]:
# ==============================================================================
# CELL 0: Dependencies Installation & Environment Setup
# ==============================================================================
!pip install -q --upgrade \
    "bitsandbytes>=0.46.1" \
    "accelerate>=0.33.0" \
    "transformers>=4.44.0" \
    "datasets>=2.20.0" \
    "tiktoken" \
    "sentencepiece" \
    "seaborn" \
    "scipy" \
    "tabulate"

import sys
import torch
import transformers
import bitsandbytes as bnb
import accelerate

print("=" * 80)
print("RM-BENCH EVALUATION ENVIRONMENT")
print("=" * 80)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers:         {transformers.__version__}")
print(f"BitsAndBytes:         {bnb.__version__}")
print(f"Accelerate:           {accelerate.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()} ({torch.cuda.device_count()} GPU(s))")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GiB")
print("=" * 80)

RM-BENCH EVALUATION ENVIRONMENT
Python Version:       3.13.16
PyTorch Version:      2.11.0+cu130 (CUDA: 13.0)
Transformers:         5.22.2
BitsAndBytes:         0.50.2
Accelerate:           1.15.0
CUDA Available:       True (2 GPU(s))
  GPU 0: Tesla T4 | VRAM: 14.56 GiB
  GPU 1: Tesla T4 | VRAM: 14.56 GiB


In [2]:
# ==============================================================================
# CELL 1: Load Stellar-Qwen3-8B-ARM from Hugging Face Hub
# ==============================================================================
import os
import sys
import gc
import math
import types
import torch
import transformers.modeling_utils
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "Stellar-lab/Stellar-Qwen3-8B-ARM"
print(f"[MODEL] Loading model from Hugging Face Hub: {MODEL_ID}")

class TiedWeightsDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, "_all_tied_weights_keys_storage", {})
    def __set__(self, instance, value):
        instance._all_tied_weights_keys_storage = value

transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = TiedWeightsDescriptor()
transformers.modeling_utils.PreTrainedModel._tied_weights_keys = {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"
print(f"[TOKENIZER] Vocab size: {len(tokenizer):,} | Pad Token ID: {tokenizer.pad_token_id}")

num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    print(f"[HARDWARE] Loading FP16 model partitioned across {num_gpus} GPUs...")
    max_memory = {i: "13.0GiB" for i in range(num_gpus)}
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
    )
elif num_gpus == 1:
    print("[HARDWARE] Single GPU detected -> Loading in 4-bit NF4...")
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_cfg,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
else:
    raise RuntimeError("A CUDA GPU is required for evaluation.")

def multi_gpu_forward_dag_head_only(self, last_6_hidden: torch.Tensor):
    head_dev = self.dag_down_proj.weight.device
    head_dtype = self.dag_down_proj.weight.dtype
    h_in = last_6_hidden.to(device=head_dev, dtype=head_dtype)

    z = self.dag_norm(self.dag_down_proj(h_in))
    for layer in self.dag_layers:
        z = layer(z, associative=self.config.associative)

    raw_logits = []
    for i in range(self.config.num_dimensions):
        head_i = self.dim_heads[i]
        h_dim = z[:, i, :].to(device=head_i.weight.device, dtype=head_i.weight.dtype)
        raw_logits.append(head_i(h_dim).to(device=head_dev))

    raw_z = torch.cat(raw_logits, dim=-1).float()
    z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)
    agg_weights = torch.nn.functional.softmax(self.agg_logits.to(device=head_dev).float(), dim=0)

    if self.config.scalar_only_mode:
        dim_logits = z_clamped
        dim_scores = torch.sigmoid(dim_logits)
        scalar_logit = (dim_logits * agg_weights).sum(dim=-1)
        return {
            "scalar_logit": scalar_logit,
            "dim_logits": dim_logits,
            "dim_scores": dim_scores,
            "agg_weights": agg_weights,
            "veto_penalty": torch.zeros_like(scalar_logit),
        }

    if self.config.associative:
        z3 = z_clamped[:, 2:3]
        z4 = z_clamped[:, 3:4]
        tau_m = self.config.monotonic_tau
        z_prereq = (
            -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True)
            + tau_m * math.log(2.0)
        )
        z5_mono = torch.clamp(
            z_prereq + self.config.monotonic_delta_tol - torch.nn.functional.softplus(-raw_z[:, 4:5]),
            min=-15.0,
            max=15.0,
        )
        dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
    else:
        dim_logits = z_clamped

    dim_scores = torch.sigmoid(dim_logits)
    linear_term = self.config.beta_lin * (dim_logits * agg_weights).sum(dim=-1)

    gk_logits = dim_logits[:, self.config.gatekeeper_indices]
    kappa = self.config.veto_kappa
    veto_lambda = torch.nn.functional.softplus(self.veto_raw_lambda.to(device=head_dev).float())
    veto_thresh = self.veto_thresholds.to(device=head_dev).float().unsqueeze(0)

    exponents = -kappa * (gk_logits - veto_thresh)
    zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=head_dev)
    veto_penalty = (veto_lambda / kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)

    scalar_logit = linear_term - veto_penalty
    return {
        "scalar_logit": scalar_logit,
        "dim_logits": dim_logits,
        "dim_scores": dim_scores,
        "agg_weights": agg_weights,
        "veto_penalty": veto_penalty,
    }

model.forward_dag_head_only = types.MethodType(multi_gpu_forward_dag_head_only, model)
model.eval()
input_device = model.backbone.get_input_embeddings().weight.device

print("=" * 80)
print(f"[MODEL READY] Loaded {model.__class__.__name__}")
for i in range(num_gpus):
    vram_gb = torch.cuda.memory_allocated(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"  GPU {i} Allocated VRAM: {vram_gb:.2f} GB / {total_gb:.2f} GB")
print("=" * 80)

[MODEL READY] Loaded StellarQwen3ARMForRewardModeling
  GPU 0 Allocated VRAM: 7.99 GB / 14.56 GB
  GPU 1 Allocated VRAM: 6.14 GB / 14.56 GB


In [3]:
# ==============================================================================
# CELL 2: Load Official `THU-KEG/RM-Bench` Dataset (1,327 Prompts x 6 Responses)
# ==============================================================================
import os
import pandas as pd
from datasets import load_dataset

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PREDICTIONS_PATH = os.path.join(OUTPUT_DIR, "rmbench_predictions.parquet")
LEADERBOARD_CSV_PATH = os.path.join(OUTPUT_DIR, "rmbench_leaderboard.csv")
LEADERBOARD_PNG_PATH = os.path.join(OUTPUT_DIR, "rmbench_comparison.png")

ds_rmb = load_dataset("THU-KEG/RM-Bench", split="train")
df_rmb = ds_rmb.to_pandas()
print(f"[RM-BENCH LOADED] Total Prompts: {len(df_rmb):,} (7,962 total completions)")

[RM-BENCH LOADED] Total Prompts: 1,327 (7,962 total completions)


In [4]:
# ==============================================================================
# CELL 3: Evaluate 3x3 Chosen-vs-Rejected Matrix Across All 1,327 Prompts
# ==============================================================================
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import List

MAX_LEN = 1280
RESP_RESERVE = 896
MICRO_BATCH_SEQS = 2

def encode_tail_safe(prompt_str: str, resp_str: str) -> List[int]:
    p_clean = str(prompt_str).strip()
    if not p_clean.startswith("User:"):
        p_clean = f"User: {p_clean}\nAssistant: "
    p_ids = tokenizer.encode(p_clean, add_special_tokens=False)
    r_ids = tokenizer.encode(str(resp_str).strip(), add_special_tokens=False)
    if len(p_ids) + len(r_ids) <= MAX_LEN:
        return p_ids + r_ids
    r_keep = r_ids[-min(len(r_ids), RESP_RESERVE):]
    p_budget = MAX_LEN - len(r_keep)
    return (p_ids[-p_budget:] if p_budget > 0 else []) + r_keep

@torch.no_grad()
def score_6_completions(prompt_str: str, resps: List[str]) -> List[float]:
    token_lists = [encode_tail_safe(prompt_str, r) for r in resps]
    pad_id = tokenizer.pad_token_id
    scores = []
    for s_idx in range(0, len(token_lists), MICRO_BATCH_SEQS):
        sub = token_lists[s_idx : s_idx + MICRO_BATCH_SEQS]
        max_l = max(len(x) for x in sub)
        padded = torch.full((len(sub), max_l), pad_id, dtype=torch.long, device=input_device)
        mask = torch.zeros((len(sub), max_l), dtype=torch.long, device=input_device)
        for i, ids in enumerate(sub):
            padded[i, max_l - len(ids):] = torch.tensor(ids, dtype=torch.long, device=input_device)
            mask[i, max_l - len(ids):] = 1
        out = model(input_ids=padded, attention_mask=mask)
        scores.extend(out.logits.float().cpu().tolist())
        del padded, mask, out
    return scores

rmbench_rows = []
for idx, row in enumerate(tqdm(df_rmb.to_dict("records"), desc="RM-Bench")):
    p = row["prompt"]
    c_list = list(row["chosen"])
    r_list = list(row["rejected"])
    domain = str(row.get("domain", str(row["id"]).split("/")[0]))

    all_scores = score_6_completions(p, c_list + r_list)
    c_sc, r_sc = all_scores[:3], all_scores[3:6]

    mat = np.zeros((3, 3), dtype=np.float64)
    for i in range(3):
        for j in range(3):
            mat[i, j] = 1.0 if c_sc[i] > r_sc[j] else (0.5 if abs(c_sc[i] - r_sc[j]) <= 1e-6 else 0.0)

    rmbench_rows.append({
        "id": str(row["id"]),
        "domain": domain,
        "chosen_scores": c_sc,
        "rejected_scores": r_sc,
        "easy_acc": float(np.sum(np.tril(mat, -1)) / 3.0),
        "normal_acc": float(np.trace(mat) / 3.0),
        "hard_acc": float(np.sum(np.triu(mat, 1)) / 3.0),
        "avg_acc": float(mat.mean()),
    })
    if (idx + 1) % 100 == 0:
        torch.cuda.empty_cache()

df_rmbench = pd.DataFrame(rmbench_rows)
df_rmbench.to_parquet(PREDICTIONS_PATH, index=False)
print(f"[SAVED] RM-Bench predictions saved to `{PREDICTIONS_PATH}`.")

[SAVED] RM-Bench predictions saved to `/kaggle/working/rmbench_predictions.parquet`.


In [5]:
# ==============================================================================
# CELL 4: RM-Bench Summary & Reference Comparison
# ==============================================================================
import pandas as pd

df_rmbench = pd.read_parquet(PREDICTIONS_PATH)
easy_acc = float(df_rmbench["easy_acc"].mean() * 100.0)
normal_acc = float(df_rmbench["normal_acc"].mean() * 100.0)
hard_acc = float(df_rmbench["hard_acc"].mean() * 100.0)
avg_acc = float(df_rmbench["avg_acc"].mean() * 100.0)

print("=" * 80)
print("RM-BENCH SUMMARY: Stellar-Qwen3-8B-ARM")
print("=" * 80)
print(f"  Easy Accuracy                 : {easy_acc:>6.2f}%")
print(f"  Normal Accuracy               : {normal_acc:>6.2f}%")
print(f"  Hard Accuracy                 : {hard_acc:>6.2f}%")
print(f"  Overall Average               : {avg_acc:>6.2f}%")
print("=" * 80)

rmb_reference = [
    {"Model": "Stellar-Qwen3-8B-ARM (Ours)", "Easy": round(easy_acc, 2), "Normal": round(normal_acc, 2), "Hard": round(hard_acc, 2), "Average": round(avg_acc, 2)},
    {"Model": "Skywork-Reward-Gemma-2-27B", "Easy": 91.60, "Normal": 74.10, "Hard": 54.00, "Average": 73.20},
    {"Model": "GPT-4o (Generative Judge)", "Easy": 85.40, "Normal": 72.10, "Hard": 52.80, "Average": 70.10},
    {"Model": "Skywork-Reward-Llama-3.1-8B", "Easy": 89.20, "Normal": 75.40, "Hard": 45.70, "Average": 70.10},
    {"Model": "ArmoRM-Llama3-8B-v0.1", "Easy": 88.00, "Normal": 72.00, "Hard": 46.80, "Average": 68.90},
    {"Model": "InternLM2-20B-Reward", "Easy": 89.50, "Normal": 71.30, "Hard": 43.80, "Average": 68.20},
]

df_rmb_lb = pd.DataFrame(rmb_reference).sort_values(by="Average", ascending=False).reset_index(drop=True)
df_rmb_lb.to_csv(LEADERBOARD_CSV_PATH, index=False)
print(df_rmb_lb.to_markdown(index=False))

RM-BENCH SUMMARY: Stellar-Qwen3-8B-ARM
  Easy Accuracy                 :  82.69%
  Normal Accuracy               :  76.44%
  Hard Accuracy                 :  65.51%
  Overall Average               :  74.88%
| Model                       |   Easy |   Normal |   Hard |   Average |
|:----------------------------|-------:|---------:|-------:|----------:|
| Stellar-Qwen3-8B-ARM (Ours) |  82.69 |    76.44 |  65.51 |     74.88 |
| Skywork-Reward-Gemma-2-27B  |  91.6  |    74.1  |  54    |     73.2  |
| GPT-4o (Generative Judge)   |  85.4  |    72.1  |  52.8  |     70.1  |
| Skywork-Reward-Llama-3.1-8B |  89.2  |    75.4  |  45.7  |     70.1  |
| ArmoRM-Llama3-8B-v0.1       |  88    |    72    |  46.8  |     68.9  |
| InternLM2-20B-Reward        |  89.5  |    71.3  |  43.8  |     68.2  |


In [6]:
# ==============================================================================
# CELL 5: Export Publication-Quality 300-DPI RM-Bench Figure
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5), dpi=300, gridspec_kw={"width_ratios": [1.2, 1.0]})

# --- Panel A: Overall Average Accuracy ---
df_plot = df_rmb_lb.sort_values(by="Average", ascending=True)
models = df_plot["Model"].tolist()
avgs = df_plot["Average"].tolist()
bar_colors = ["#d95f02" if "Ours" in m else ("#4a6fe3" if "8B" in m else "#8c8c8c") for m in models]

bars = ax1.barh(models, avgs, color=bar_colors, height=0.62, edgecolor="black", linewidth=0.7)
ax1.axvline(50.0, color="red", linestyle=":", linewidth=1.2, label="Random Baseline (50.0%)")
ax1.set_xlim(45, 85)
ax1.set_xlabel("Overall Average Accuracy (%)", fontsize=11, fontweight="bold")
ax1.set_title("A. RM-Bench: Overall Average Accuracy", fontsize=11, fontweight="bold")
ax1.grid(axis="x", linestyle="--", alpha=0.45)
ax1.legend(loc="lower right", fontsize=9)

for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.4, bar.get_y() + bar.get_height() / 2, f"{w:.2f}%", va="center", ha="left", fontsize=9, fontweight="bold")

# --- Panel B: Easy / Normal / Hard Breakdown ---
splits = ["Easy", "Normal", "Hard"]
x = np.arange(len(splits))
width = 0.25

ours_r = df_rmb_lb[df_rmb_lb["Model"].str.contains("Ours")].iloc[0]
sky_8b = df_rmb_lb[df_rmb_lb["Model"].str.contains("Skywork-Reward-Llama-3.1-8B")].iloc[0]
armo_r = df_rmb_lb[df_rmb_lb["Model"].str.contains("ArmoRM")].iloc[0]

ax2.bar(x - width, [ours_r[s] for s in splits], width, label="Stellar-Qwen3-8B-ARM (Ours)", color="#d95f02", edgecolor="black")
ax2.bar(x,         [sky_8b[s] for s in splits], width, label="Skywork-Reward-Llama-3.1-8B", color="#4a6fe3", edgecolor="black")
ax2.bar(x + width, [armo_r[s] for s in splits], width, label="ArmoRM-Llama3-8B-v0.1", color="#8c8c8c", edgecolor="black")

ax2.set_xticks(x)
ax2.set_xticklabels(splits, fontsize=10, fontweight="bold")
ax2.set_ylim(30, 100)
ax2.set_ylabel("Accuracy (%)", fontsize=11, fontweight="bold")
ax2.set_title("B. Accuracy by Style Difficulty Split (8B Models)", fontsize=11, fontweight="bold")
ax2.grid(axis="y", linestyle="--", alpha=0.45)
ax2.legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
fig.savefig(LEADERBOARD_PNG_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[PLOT SAVED] -> {LEADERBOARD_PNG_PATH}")
display(Image(filename=LEADERBOARD_PNG_PATH))

[PLOT SAVED] -> /kaggle/working/rmbench_comparison.png


### Comparison Figure

![Stellar-Qwen3-8B-ARM — RM-Bench Evaluation](assets/rmbench_comparison.png)